# Data Pipeline – Hướng dẫn Config, Updater & Query

Notebook này hướng dẫn **3 module cốt lõi** của hệ thống dữ liệu `vnstock_forecast`:

| Module | File | Vai trò |
|---|---|---|
| **Config** | `schemas/`, `utils/config_utils.py` | Load & validate cấu hình YAML (Hydra/OmegaConf) |
| **Updater** | `data/updater.py` | Tải OHLCV từ Vietstock API → lưu Parquet (incremental) |
| **Query** | `data/query.py` | Truy vấn DuckDB trên Parquet store |

**Không cần dữ liệu thật cho phần Config & Query demo** — ta sẽ tạo Parquet giả lập.

---

## Kiến trúc Data Pipeline

```
┌───────────────────────────────────────────────────────────────────────┐
│                        Config (Hydra/OmegaConf)                       │
│                                                                       │
│  config/config.yaml ──► load_config() ──► AppConfig (dataclass)      │
│       ├── data.discovery.symbols.vn30: [SHB, VHM]                    │
│       ├── data.discovery.resolutions.vietstock: {daily: D, m1: 1}    │
│       └── data.updater: {client, symbols, resolutions, lookback}     │
└───────────────┬───────────────────────────────────────────────────────┘
                │
                v
┌───────────────────────────────────────────────────────────────────────┐
│                        Updater (data/updater.py)                      │
│                                                                       │
│  update(cfg.data.updater)                                             │
│    ├─ Vietstock API ──► fetch OHLCV                                  │
│    ├─ Merge & deduplicate                                             │
│    └─ Save → data/ohlcv/resolution=D/VHM.parquet                    │
└───────────────┬───────────────────────────────────────────────────────┘
                │
                v
┌───────────────────────────────────────────────────────────────────────┐
│                        Query (data/query.py)                          │
│                                                                       │
│  data/ohlcv/                                                          │
│    resolution=D/VHM.parquet ─┐                                       │
│    resolution=D/SHB.parquet ─┼──► DuckDB (hive partitioning)        │
│    resolution=W/VHM.parquet ─┘       │                               │
│                                      v                                │
│                              query_ohlcv()                            │
│                              query_latest()                           │
│                              query_ohlcv_grouped()                   │
│                              query_sql()                              │
└───────────────────────────────────────────────────────────────────────┘
```

## 1. Import & Setup

Đảm bảo package đã được cài (`pip install -e .` từ root project).  
Nếu chạy notebook từ thư mục `notebooks/`, đoạn code dưới sẽ tự thêm `src/` vào path.

In [2]:
# ── Config utilities ─────────────────────────────────────────────────
from vnstock_forecast.utils.config_utils import load_config, print_config, get_project_root
from vnstock_forecast.schemas import AppConfig, UpdaterConfig
from vnstock_forecast.shared.path import CONFIG_PATH_STR, DATA_PATH_STR

# ── Data modules ─────────────────────────────────────────────────────
from vnstock_forecast.data.query import (
    query_ohlcv,
    query_latest,
    query_ohlcv_grouped,
    query_sql,
)
from vnstock_forecast.data.updater import update, update_symbol, OHLCV_BASE_DIR
from vnstock_forecast.utils import time_utils

# ── Thư viện ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from omegaconf import OmegaConf

print(f"Project root : {get_project_root()}")
print(f"Config path  : {CONFIG_PATH_STR}")
print(f"Data path    : {DATA_PATH_STR}")
print(f"OHLCV store  : {OHLCV_BASE_DIR}")
print("\n✓ Import thành công")

Project root : /mnt/data/QuickAccess/Documents/4.Tin/Projects/vnstock-forecast
Config path  : /mnt/data/QuickAccess/Documents/4.Tin/Projects/vnstock-forecast/config
Data path    : /mnt/data/QuickAccess/Documents/4.Tin/Projects/vnstock-forecast/data
OHLCV store  : /mnt/data/QuickAccess/Documents/4.Tin/Projects/vnstock-forecast/data/ohlcv

✓ Import thành công


---
## 2. Config System (Hydra + OmegaConf)

Dự án dùng [Hydra](https://hydra.cc/) + [OmegaConf](https://omegaconf.readthedocs.io/) để quản lý cấu hình:

- **YAML files** trong `config/` → định nghĩa giá trị
- **Dataclass schemas** trong `schemas/` → validate kiểu dữ liệu
- **`load_config()`** → compose tất cả YAML thành 1 object duy nhất

### Cấu trúc thư mục config

```
config/
├── config.yaml                          # Entry-point chính
├── data/
│   ├── updater.yaml                     # Cấu hình cho data updater
│   └── discovery/
│       ├── symbols/
│       │   ├── vn30.yaml                # Danh sách mã VN30
│       │   └── vnindex.yaml             # Danh sách mã VNIndex
│       └── resolutions/
│           └── vietstock.yaml           # Mapping resolution cho Vietstock API
```

### Cơ chế hoạt động

1. `config.yaml` khai báo `defaults:` để Hydra merge các file YAML con
2. Hydra compose tất cả → 1 `DictConfig`
3. `OmegaConf.to_object()` chuyển thành dataclass `AppConfig` (type-safe)

In [3]:
# ── 2.1 Load config dạng DictConfig (raw) ────────────────────────────
cfg = load_config()

print("Kiểu trả về:", type(cfg))
print("\n── Toàn bộ config (YAML) ──────────────────────────────────────")
print_config(cfg)

Kiểu trả về: <class 'omegaconf.dictconfig.DictConfig'>

── Toàn bộ config (YAML) ──────────────────────────────────────
data:
  discovery:
    symbols:
      vn30:
      - SHB
      - VHM
      vnindex:
      - HPG
      - VNM
    resolutions:
      vietstock:
        daily: D
        m1: 1
        m5: 5
        m15: 15
        m30: 30
        m45: 45
        h1: 60
        h3: 180
        h4: 240
  updater:
    client: vietstock
    symbols:
    - SHB
    - VHM
    resolutions:
    - D
    lookback_days: 300



In [4]:
# ── 2.2 Truy cập từng phần config ─────────────────────────────────────
# Config có cấu trúc cây — truy cập bằng dot notation hoặc dict-style

# Danh sách mã VN30
print("Symbols VN30  :", list(cfg.data.discovery.symbols.vn30))
print("Symbols VNIndex:", list(cfg.data.discovery.symbols.vnindex))
print()

# Resolution mapping cho Vietstock API
res = cfg.data.discovery.resolutions.vietstock
print("Resolution mapping (Vietstock):")
print(f"  daily = '{res.daily}'")
print(f"  m1    = {res.m1}")
print(f"  m15   = {res.m15}")
print(f"  h1    = {res.h1}")
print(f"  h4    = {res.h4}")
print()

# Cấu hình updater
updater_cfg = cfg.data.updater
print("Updater config:")
print(f"  client        = {updater_cfg.client}")
print(f"  symbols       = {list(updater_cfg.symbols)}")
print(f"  resolutions   = {list(updater_cfg.resolutions)}")
print(f"  lookback_days = {updater_cfg.lookback_days}")

Symbols VN30  : ['SHB', 'VHM']
Symbols VNIndex: ['HPG', 'VNM']

Resolution mapping (Vietstock):
  daily = 'D'
  m1    = 1
  m15   = 15
  h1    = 60
  h4    = 240

Updater config:
  client        = DataClient.vietstock
  symbols       = ['SHB', 'VHM']
  resolutions   = ['D']
  lookback_days = 300


In [5]:
# ── 2.3 Override config tại runtime ───────────────────────────────────
# Hydra cho phép override bất kỳ giá trị nào qua danh sách overrides.
# Rất hữu ích khi thử nghiệm mà không sửa file YAML.

cfg_custom = load_config(overrides=[
    "data.updater.lookback_days=60",
    "data.updater.symbols=[FPT,MWG]",
    "data.updater.resolutions=[D,W]",
])

print("── Config sau override ─────────────────────────────────────")
print(f"  symbols       = {list(cfg_custom.data.updater.symbols)}")
print(f"  resolutions   = {list(cfg_custom.data.updater.resolutions)}")
print(f"  lookback_days = {cfg_custom.data.updater.lookback_days}")

── Config sau override ─────────────────────────────────────
  symbols       = ['FPT', 'MWG']
  resolutions   = ['D', 'W']
  lookback_days = 60


In [6]:
# ── 2.4 Chuyển DictConfig → Python object thuần (AppConfig dataclass) ─
from vnstock_forecast.schemas.config import load_app_config

app_cfg = load_app_config()

print("Kiểu trả về:", type(app_cfg))
print(f"  app_cfg.data.updater.client        = {app_cfg.data.updater.client}")
print(f"  app_cfg.data.updater.symbols       = {app_cfg.data.updater.symbols}")
print(f"  app_cfg.data.updater.lookback_days = {app_cfg.data.updater.lookback_days}")
print()
print("Lưu ý: load_app_config() trả về Python dict/list thuần,")
print("        không còn là OmegaConf DictConfig → dùng để truyền vào các hàm Python.")

Kiểu trả về: <class 'vnstock_forecast.schemas.config.AppConfig'>
  app_cfg.data.updater.client        = DataClient.vietstock
  app_cfg.data.updater.symbols       = ['SHB', 'VHM']
  app_cfg.data.updater.lookback_days = 300

Lưu ý: load_app_config() trả về Python dict/list thuần,
        không còn là OmegaConf DictConfig → dùng để truyền vào các hàm Python.


### Tổng kết Config

| Hàm | Trả về | Khi nào dùng |
|---|---|---|
| `load_config()` | `DictConfig` (OmegaConf) | Cần interpolation, override, merge |
| `load_app_config()` | `AppConfig` (Python dict thuần) | Truyền vào hàm Python, serialize |
| `print_config(cfg)` | In YAML | Debug, kiểm tra config |

> **Tip**: Dùng `load_config(overrides=[...])` để thử nghiệm nhanh mà không sửa file YAML.

---
## 3. Schema – Cấu trúc dữ liệu cấu hình

Hydra sử dụng `@dataclass` để validate kiểu dữ liệu. Dưới đây là sơ đồ các schema:

```
AppConfig                          ← schemas/config.py
└── data: DataConfig               ← schemas/data.py
    ├── discovery: DiscoveryConfig
    │   ├── symbols: DiscoverySymbolsConfig
    │   │   ├── vn30: list[str]           → ["SHB", "VHM"]
    │   │   └── vnindex: list[str]        → ["HPG", "VNM"]
    │   └── resolutions: DiscoveryResolutionConfig
    │       └── vietstock: VietstockResolutionConfig
    │           ├── daily: str = "D"
    │           ├── m1: int = 1
    │           ├── m15: int = 15
    │           ├── h1: int = 60
    │           └── h4: int = 240
    └── updater: UpdaterConfig
        ├── client: DataClient            → "vietstock"
        ├── symbols: list[str]            → (từ discovery — interpolation)
        ├── resolutions: list[str]        → ["D"]
        └── lookback_days: int = 365
```

> **Interpolation**: Trong `updater.yaml`, trường `symbols: ${data.discovery.symbols.vn30}` sẽ tự động resolve sang danh sách `["SHB", "VHM"]` nhờ OmegaConf.

In [7]:
# ── 3.1 Xem schema của từng dataclass ─────────────────────────────────
from dataclasses import fields
from vnstock_forecast.schemas.data import (
    DataConfig,
    DiscoveryConfig,
    DiscoverySymbolsConfig,
    VietstockResolutionConfig,
    UpdaterConfig,
    DataClient,
)

schemas = [
    ("AppConfig", AppConfig),
    ("DataConfig", DataConfig),
    ("UpdaterConfig", UpdaterConfig),
    ("DiscoverySymbolsConfig", DiscoverySymbolsConfig),
    ("VietstockResolutionConfig", VietstockResolutionConfig),
]

for name, cls in schemas:
    print(f"── {name} ──")
    for f in fields(cls):
        print(f"  {f.name:20s} : {f.type}")
    print()

── AppConfig ──
  data                 : <class 'vnstock_forecast.schemas.data.DataConfig'>

── DataConfig ──
  discovery            : <class 'vnstock_forecast.schemas.data.DiscoveryConfig'>
  updater              : <class 'vnstock_forecast.schemas.data.UpdaterConfig'>

── UpdaterConfig ──
  client               : <enum 'DataClient'>
  symbols              : list[str]
  resolutions          : list[str]
  lookback_days        : <class 'int'>

── DiscoverySymbolsConfig ──
  vn30                 : list[str]
  vnindex              : list[str]

── VietstockResolutionConfig ──
  daily                : <class 'str'>
  m1                   : <class 'int'>
  m5                   : <class 'int'>
  m15                  : <class 'int'>
  m30                  : <class 'int'>
  m45                  : <class 'int'>
  h1                   : <class 'int'>
  h3                   : <class 'int'>
  h4                   : <class 'int'>



---
## 4. Updater – Cập nhật dữ liệu từ Vietstock API

Module `data/updater.py` thực hiện:

1. **Kiểm tra dữ liệu local** (Parquet) — nếu đã có, chỉ fetch data mới hơn (incremental)
2. **Backfill** — nếu `lookback_days` lớn hơn dữ liệu local, fetch thêm phần thiếu
3. **Merge & deduplicate** — gộp data cũ + mới, loại trùng
4. **Lưu Parquet** — ghi vào `data/ohlcv/resolution=<res>/<SYMBOL>.parquet`

### Storage Layout (Hive-style partitioning)

```
data/ohlcv/
├── resolution=D/
│   ├── VHM.parquet          ← mỗi file = 1 (symbol, resolution)
│   └── SHB.parquet
├── resolution=W/
│   ├── VHM.parquet
│   └── SHB.parquet
├── resolution=1/
│   └── ...
└── resolution=60/
    └── ...
```

Mỗi file `.parquet` chứa các cột: `Timestamp | Symbol | Open | High | Low | Close | Volume`

### API chính

| Hàm | Mô tả |
|---|---|
| `update_symbol(client, symbol, resolution, lookback_days)` | Cập nhật 1 cặp (symbol, resolution) |
| `update(cfg: UpdaterConfig)` | Chạy toàn bộ theo config (symbol × resolution) |

In [8]:
# ── 4.1 Cách dùng update() với config ────────────────────────────────
# ⚠️ Cell này KHÔNG chạy thật (cần kết nối internet + Vietstock API).
# Chỉ minh hoạ cách sử dụng.

# --- Cách 1: Dùng config mặc định từ YAML ---
# cfg = load_config()
# update(cfg.data.updater)             # Fetch all symbols × resolutions trong config

# --- Cách 2: Override config ---
# cfg = load_config(overrides=[
#     "data.updater.symbols=[FPT,MWG,VNM]",
#     "data.updater.resolutions=[D,W]",
#     "data.updater.lookback_days=180",
# ])
# update(cfg.data.updater)

# --- Cách 3: Update từng symbol riêng lẻ ---
# from vnstock_forecast.client.vietstock import OHLCV
# client = OHLCV()
# update_symbol(client, symbol="FPT", resolution="D", lookback_days=90)

print("⚠️ Cell này chỉ minh hoạ — bỏ comment để chạy thật với API.")

⚠️ Cell này chỉ minh hoạ — bỏ comment để chạy thật với API.


### Luồng xử lý chi tiết của `update_symbol()`

```
update_symbol(client, "VHM", "D", lookback_days=300)
│
├─ Đọc file: data/ohlcv/resolution=D/VHM.parquet
│   ├─ Nếu tồn tại → lấy local_min_ts, local_max_ts
│   └─ Nếu không tồn tại → full fetch
│
├─ Backfill (nếu lookback > local_min):
│   └─ fetch(lookback_ts → local_min_ts - 1)
│
├─ Forward (nếu local_max < now):
│   └─ fetch(local_max_ts + 1 → now)
│
├─ Merge & deduplicate (drop duplicates on Timestamp + Symbol)
│
└─ Save → data/ohlcv/resolution=D/VHM.parquet
```

> **Incremental**: Chỉ fetch dữ liệu thiếu, không download lại toàn bộ → tiết kiệm thời gian & bandwidth.

---
## 5. Time Utilities

Module `utils/time_utils.py` cung cấp các helper chuyển đổi thời gian — cần thiết vì Vietstock API sử dụng **Unix timestamp** (giây).

| Hàm | Mô tả |
|---|---|
| `time_to_timestamp("2024-01-15")` | Chuyển str/datetime → Unix timestamp |
| `timestamp_to_str(1705276800)` | Unix timestamp → chuỗi ngày giờ |
| `get_current_timestamp()` | Lấy timestamp hiện tại |
| `get_current_date_timestamp()` | Timestamp nửa đêm hôm nay |
| `add_days_to_timestamp(ts, -30)` | Cộng/trừ ngày |

In [9]:
# ── 5.1 Demo time_utils ───────────────────────────────────────────────

# String → Timestamp
ts = time_utils.time_to_timestamp("2024-06-15")
print(f"'2024-06-15' → timestamp = {ts}")

# Timestamp → String
print(f"timestamp {ts} → '{time_utils.timestamp_to_str(ts)}'")

# Timestamp hiện tại
now_ts = time_utils.get_current_timestamp()
print(f"\nTimestamp hiện tại: {now_ts}  ({time_utils.timestamp_to_str(now_ts)})")

# Trừ 30 ngày
ts_30d_ago = time_utils.add_days_to_timestamp(now_ts, -30)
print(f"30 ngày trước    : {ts_30d_ago}  ({time_utils.timestamp_to_str(ts_30d_ago)})")

# Timestamp milliseconds
ts_ms = time_utils.time_to_timestamp("2024-06-15", unit="ms")
print(f"\nMillisecond mode : {ts_ms}")

'2024-06-15' → timestamp = 1718409600
timestamp 1718409600 → '2024-06-15 00:00:00'

Timestamp hiện tại: 1772988152  (2026-03-08 16:42:32)
30 ngày trước    : 1770396152  (2026-02-06 16:42:32)

Millisecond mode : 1718409600000
